# 03 Scoring and H3 Export
This notebook creates the universal baseline score, the Track A Healthy Lifestyle & Sport score, and the composite score. Results are aggregated to Uber H3 resolution 8 and exported as GeoJSON for the Web GIS app.

## 0. Setup

In [ ]:
from pathlib import Path
import shutil
import sys
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from fifteenmc.paths import load_project_paths, load_yaml
from fifteenmc.scoring import add_scores
from fifteenmc.h3_export import aggregate_to_h3, export_frontend_geojson
from fifteenmc.validation import assert_score_columns, assert_geojson_loads

paths = load_project_paths(PROJECT_ROOT)
weights = load_yaml(PROJECT_ROOT / 'config' / 'weights.yaml')

## 1. Weighting Rationale
The baseline gives comparable weight to food, healthcare, education, transit, parks, and daily services. Track A prioritizes gyms, public exercise parks, courts/fields, pools, studios, cycling infrastructure, fresh markets, greenery, and AQI. Weights are editable in `config/weights.yaml`. Walk and bike are the official 15MC modes; transit and car are comparison layers.

In [ ]:
weights

## 2. Load Grid Accessibility Metrics

In [ ]:
access = pd.read_parquet(paths.output('grid_accessibility'))
print(access.shape)
access.head()

## 3. Calculate Baseline, Track A, and Composite Scores

In [ ]:
scores = add_scores(access, weights)
assert_score_columns(scores)
scores.to_parquet(paths.output('grid_scores'), index=False)
print(paths.output('grid_scores'))
scores[['grid_id', 'mode', 'baseline_score', 'track_score', 'composite_score']].head()

## 4. Score Distribution Check

In [ ]:
scores.groupby('mode')[['baseline_score', 'track_score', 'composite_score']].describe().round(2)

## 5. Aggregate to Uber H3 Resolution 8

In [ ]:
h3_scores = aggregate_to_h3(scores, resolution=paths.config['analysis']['h3_resolution'])
h3_scores.to_parquet(paths.output('h3_scores'), index=False)
print(h3_scores.shape)
h3_scores.head()

## 6. Export Web GeoJSON

In [ ]:
export_frontend_geojson(h3_scores, paths.output('h3_geojson'))
shutil.copy2(paths.output('h3_geojson'), paths.output('app_h3_geojson'))
assert_geojson_loads(paths.output('h3_geojson'))
print(paths.output('h3_geojson'))
print(paths.output('app_h3_geojson'))

## 7. Interpretation Notes
The exported GeoJSON includes `data_quality_flags` to make the approximation visible in the app. If true routing, NDVI, or AQI data are added, rerun this notebook to refresh H3 scores.